In [1]:
import os, glob, shutil
from pathlib import Path

model_path = "efficientNet_best.pth"

search_roots = [
    "../input",  # classic notebook relative path
    "/kaggle/input",  # Kaggle standard absolute path
    "/kaggle/data/input",  # present in this environment listing
    "/kaggle/data",  # present in this environment listing
    "/kaggle/data/input/aptos2019-blindness-detection",  # explicit competition root (robust)
    "/kaggle/data/aptos2019-blindness-detection",  # explicit competition root (robust)
]

candidate_patterns = []
for r in search_roots:
    candidate_patterns.extend(
        [
            f"{r}/**/efficientnet*_best*.pth",
            f"{r}/**/efficientnet*best*.pth",
            f"{r}/**/efficientnet*.pth",
            f"{r}/**/efficientNet*_best*.pth",
            f"{r}/**/efficientNet*best*.pth",
            f"{r}/**/efficientNet*.pth",
            f"{r}/**/*.pth",
        ]
    )

candidates = []
for pat in candidate_patterns:
    candidates.extend(glob.glob(pat, recursive=True))


def _score_name(p):
    name = os.path.basename(p).lower()
    score = 0
    if "efficientnet" in name or "efficient" in name:
        score += 60
    if "best" in name:
        score += 30
    if "fold" in name:
        score += 3
    if "final" in name:
        score += 5
    if "optim" in name or "optimizer" in name:
        score -= 80
    if "sched" in name or "scheduler" in name:
        score -= 40
    if "ema" in name:
        score -= 5
    if "epoch" in name and "best" not in name:
        score -= 3
    if name.endswith(".pth"):
        score += 1
    try:
        sz = os.path.getsize(p)
    except OSError:
        sz = 0
    if sz > 80_000_000:
        score += 35
    elif sz > 50_000_000:
        score += 25
    elif sz > 10_000_000:
        score += 10
    elif sz > 1_000_000:
        score += 2
    else:
        score -= 10
    return score


def _sort_key(p):
    try:
        sz = os.path.getsize(p)
    except OSError:
        sz = 0
    return (-_score_name(p), -sz, p)


candidates = sorted(set(candidates), key=_sort_key)
print("Found .pth candidates (top 25 by heuristic):")
for p in candidates[:25]:
    try:
        sz = os.path.getsize(p)
    except OSError:
        sz = -1
    print(" -", p, "score=", _score_name(p), "size=", sz)

print(
    "model_path exists?",
    os.path.exists(model_path),
    "size:",
    os.path.getsize(model_path) if os.path.exists(model_path) else None,
)



Found .pth candidates (top 25 by heuristic):
model_path exists? False size: None


In [2]:
import torch
import torch.nn as nn


class Swish(nn.Module):
    def forward(self, x):
        return x * torch.sigmoid(x)


class Flatten(nn.Module):
    def forward(self, x):
        return x.reshape(x.shape[0], -1)


class SqueezeExcitation(nn.Module):

    def __init__(self, inplanes, se_planes):
        super(SqueezeExcitation, self).__init__()
        self.reduce_expand = nn.Sequential(
            nn.Conv2d(
                inplanes, se_planes, kernel_size=1, stride=1, padding=0, bias=True
            ),
            Swish(),
            nn.Conv2d(
                se_planes, inplanes, kernel_size=1, stride=1, padding=0, bias=True
            ),
            nn.Sigmoid(),
        )

    def forward(self, x):
        x_se = torch.mean(x, dim=(-2, -1), keepdim=True)
        x_se = self.reduce_expand(x_se)
        return x_se * x


from torch.nn import functional as F


class MBConv(nn.Module):
    def __init__(
        self,
        inplanes,
        planes,
        kernel_size,
        stride,
        expand_rate=1.0,
        se_rate=0.25,
        drop_connect_rate=0.2,
    ):
        super(MBConv, self).__init__()

        expand_planes = int(inplanes * expand_rate)
        se_planes = max(1, int(inplanes * se_rate))

        self.expansion_conv = None
        if expand_rate > 1.0:
            self.expansion_conv = nn.Sequential(
                nn.Conv2d(
                    inplanes,
                    expand_planes,
                    kernel_size=1,
                    stride=1,
                    padding=0,
                    bias=False,
                ),
                nn.BatchNorm2d(expand_planes, momentum=0.01, eps=1e-3),
                Swish(),
            )
            inplanes = expand_planes

        self.depthwise_conv = nn.Sequential(
            nn.Conv2d(
                inplanes,
                expand_planes,
                kernel_size=kernel_size,
                stride=stride,
                padding=kernel_size // 2,
                groups=expand_planes,
                bias=False,
            ),
            nn.BatchNorm2d(expand_planes, momentum=0.01, eps=1e-3),
            Swish(),
        )

        self.squeeze_excitation = SqueezeExcitation(expand_planes, se_planes)

        self.project_conv = nn.Sequential(
            nn.Conv2d(
                expand_planes, planes, kernel_size=1, stride=1, padding=0, bias=False
            ),
            nn.BatchNorm2d(planes, momentum=0.01, eps=1e-3),
        )

        self.with_skip = stride == 1

        # Change (stability/score): keep drop_connect_rate as a python float to avoid device mismatch
        # when the model is on CUDA and drop-connect is activated during training.
        self.drop_connect_rate = float(drop_connect_rate)

    def _drop_connect(self, x):
        keep_prob = 1.0 - self.drop_connect_rate
        drop_mask = torch.rand(x.shape[0], 1, 1, 1, device=x.device) + keep_prob
        drop_mask = drop_mask.type_as(x)
        drop_mask.floor_()
        return drop_mask * x / keep_prob

    def forward(self, x):
        z = x
        if self.expansion_conv is not None:
            x = self.expansion_conv(x)

        x = self.depthwise_conv(x)
        x = self.squeeze_excitation(x)
        x = self.project_conv(x)

        if x.shape == z.shape and self.with_skip:
            if self.training and self.drop_connect_rate is not None:
                x = self._drop_connect(x)
            x += z
        return x


from collections import OrderedDict
import math


def init_weights(module):
    if isinstance(module, nn.Conv2d):
        nn.init.kaiming_normal_(module.weight, a=0, mode="fan_out")
    elif isinstance(module, nn.Linear):
        init_range = 1.0 / math.sqrt(module.weight.shape[1])
        nn.init.uniform_(module.weight, a=-init_range, b=init_range)


class EfficientNet(nn.Module):

    def _setup_repeats(self, num_repeats):
        return int(math.ceil(self.depth_coefficient * num_repeats))

    def _setup_channels(self, num_channels):
        num_channels *= self.width_coefficient
        new_num_channels = math.floor(num_channels / self.divisor + 0.5) * self.divisor
        new_num_channels = max(self.divisor, new_num_channels)
        if new_num_channels < 0.9 * num_channels:
            new_num_channels += self.divisor
        return new_num_channels

    def __init__(
        self,
        num_classes,
        width_coefficient=1.0,
        depth_coefficient=1.0,
        se_rate=0.25,
        dropout_rate=0.2,
        drop_connect_rate=0.2,
    ):
        super(EfficientNet, self).__init__()

        self.width_coefficient = width_coefficient
        self.depth_coefficient = depth_coefficient
        self.divisor = 8

        list_channels = [32, 16, 24, 40, 80, 112, 192, 320, 1280]
        list_channels = [self._setup_channels(c) for c in list_channels]

        list_num_repeats = [1, 2, 2, 3, 3, 4, 1]
        list_num_repeats = [self._setup_repeats(r) for r in list_num_repeats]

        expand_rates = [1, 6, 6, 6, 6, 6, 6]
        strides = [1, 2, 2, 2, 1, 2, 1]
        kernel_sizes = [3, 3, 5, 3, 5, 5, 3]

        self.stem = nn.Sequential(
            nn.Conv2d(
                3, list_channels[0], kernel_size=3, stride=2, padding=1, bias=False
            ),
            nn.BatchNorm2d(list_channels[0], momentum=0.01, eps=1e-3),
            Swish(),
        )

        blocks = []
        counter = 0
        num_blocks = sum(list_num_repeats)
        for idx in range(7):

            num_channels = list_channels[idx]
            next_num_channels = list_channels[idx + 1]
            num_repeats = list_num_repeats[idx]
            expand_rate = expand_rates[idx]
            kernel_size = kernel_sizes[idx]
            stride = strides[idx]
            drop_rate = drop_connect_rate * counter / num_blocks

            name = "MBConv{}_{}".format(expand_rate, counter)
            blocks.append(
                (
                    name,
                    MBConv(
                        num_channels,
                        next_num_channels,
                        kernel_size=kernel_size,
                        stride=stride,
                        expand_rate=expand_rate,
                        se_rate=se_rate,
                        drop_connect_rate=drop_rate,
                    ),
                )
            )
            counter += 1
            for i in range(1, num_repeats):
                name = "MBConv{}_{}".format(expand_rate, counter)
                drop_rate = drop_connect_rate * counter / num_blocks
                blocks.append(
                    (
                        name,
                        MBConv(
                            next_num_channels,
                            next_num_channels,
                            kernel_size=kernel_size,
                            stride=1,
                            expand_rate=expand_rate,
                            se_rate=se_rate,
                            drop_connect_rate=drop_rate,
                        ),
                    )
                )
                counter += 1

        self.blocks = nn.Sequential(OrderedDict(blocks))

        self.head = nn.Sequential(
            nn.Conv2d(list_channels[-2], list_channels[-1], kernel_size=1, bias=False),
            nn.BatchNorm2d(list_channels[-1], momentum=0.01, eps=1e-3),
            Swish(),
            nn.AdaptiveAvgPool2d(1),
            Flatten(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(list_channels[-1], num_classes),
        )

        self.apply(init_weights)

    def forward(self, x):
        f = self.stem(x)
        f = self.blocks(f)
        y = self.head(f)
        return y




In [3]:
import os
import shutil
import random
import numpy as np
import torch
from collections import OrderedDict

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

torch.manual_seed(0)
np.random.seed(0)
random.seed(0)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(0)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

best_model = EfficientNet(num_classes=5).to(device)
best_model.eval()


def _extract_state_dict(ckpt_obj):
    if isinstance(ckpt_obj, dict):
        for k in ["state_dict", "model_state_dict", "model", "net", "weights"]:
            if k in ckpt_obj and isinstance(ckpt_obj[k], dict):
                return ckpt_obj[k]
    return ckpt_obj


def _strip_known_prefixes(state):
    prefixes = ("module.", "model.", "net.", "encoder.", "backbone.")
    new_state = OrderedDict()
    for k, v in state.items():
        nk = k
        changed = True
        while changed:
            changed = False
            for pref in prefixes:
                if nk.startswith(pref):
                    nk = nk[len(pref) :]
                    changed = True
        new_state[nk] = v
    return new_state


def _try_load_checkpoint(path, model):
    try:
        ckpt = torch.load(path, map_location="cpu")
    except Exception as e:
        print(f"Could not load checkpoint {path}: {type(e).__name__}: {e}")
        return 0.0, False, None

    state = _extract_state_dict(ckpt)
    if not isinstance(state, dict):
        print(f"Checkpoint {path} is not state_dict-like; skipping.")
        return 0.0, False, None

    state = _strip_known_prefixes(state)

    model_keys = set(model.state_dict().keys())
    state_keys = set(state.keys())
    overlap = len(model_keys & state_keys)
    loaded_frac = overlap / max(1, len(model_keys))

    missing, unexpected = model.load_state_dict(state, strict=False)
    print(
        f"Try load: {path} | overlap={overlap}/{len(model_keys)} ({loaded_frac:.3f}) | missing={len(missing)} unexpected={len(unexpected)}"
    )
    return loaded_frac, True, state


cand_list = []
if "candidates" in globals():
    cand_list = list(candidates)

if os.path.exists(model_path):
    cand_list = [model_path] + [
        p for p in cand_list if os.path.abspath(p) != os.path.abspath(model_path)
    ]

cand_list = list(dict.fromkeys(cand_list))[:80]

best_loaded_frac = -1.0
best_sz = -1
best_path = None
best_state = None

for p in cand_list:
    if not os.path.exists(p):
        continue
    frac, ok, state = _try_load_checkpoint(p, best_model)
    if not ok:
        continue
    try:
        sz = os.path.getsize(p)
    except OSError:
        sz = 0
    if (frac > best_loaded_frac) or (frac == best_loaded_frac and sz > best_sz):
        best_loaded_frac = frac
        best_sz = sz
        best_path = p
        best_state = state

have_external_weights = best_path is not None and best_loaded_frac >= 0.50
print(
    "have_external_weights:",
    have_external_weights,
    "| best_path:",
    best_path,
    "| overlap:",
    best_loaded_frac,
)

if have_external_weights:
    print(
        f"Selected checkpoint: {best_path} (overlap={best_loaded_frac:.3f}, size={best_sz})"
    )

    if os.path.abspath(best_path) != os.path.abspath(model_path):
        try:
            shutil.copy(best_path, model_path)
            print(f"Copied selected weights -> {model_path}")
        except Exception as e:
            print(f"WARNING: could not copy selected weights to {model_path}: {e}")

    best_model = EfficientNet(num_classes=5).to(device)
    best_model.eval()
    missing, unexpected = best_model.load_state_dict(best_state, strict=False)
    print(f"Final load missing={len(missing)} unexpected={len(unexpected)}")
else:
    print(
        "No suitable external checkpoint found; will train a small model from scratch on train_images for a valid (non-random) submission."
    )



device: cuda
have_external_weights: False | best_path: None | overlap: -1.0
No suitable external checkpoint found; will train a small model from scratch on train_images for a valid (non-random) submission.


In [4]:
from torchvision.transforms import Compose, Resize
from torchvision.transforms import ToTensor, Normalize

import pandas as pd
from PIL import Image
from PIL.Image import BICUBIC


class ImageDataset(torch.utils.data.Dataset):

    def __init__(self, root, path_list, targets=None, transform=None, extension=".png"):
        super().__init__()
        self.root = root
        self.path_list = list(path_list)
        self.targets = targets
        self.transform = transform
        self.extension = extension
        if targets is not None:
            assert len(self.path_list) == len(self.targets)
            self.targets = torch.LongTensor(targets)

    def __getitem__(self, index):
        path = self.path_list[index]
        img_path = os.path.join(self.root, path + self.extension)
        sample = Image.open(img_path).convert("RGB")
        if self.transform is not None:
            sample = self.transform(sample)

        if self.targets is not None:
            return sample, self.targets[index]
        else:
            return sample, torch.LongTensor([])

    def __len__(self):
        return len(self.path_list)


image_size = 224

test_transform = Compose(
    [
        Resize((image_size, image_size), BICUBIC),
        ToTensor(),
        Normalize(mean=[0.42, 0.22, 0.075], std=[0.27, 0.15, 0.081]),
    ]
)

test_csv_path = "../input/aptos2019-blindness-detection/test.csv"
test_img_root = "../input/aptos2019-blindness-detection/test_images"
if not os.path.exists(test_csv_path):
    for alt_root in ["/kaggle/input", "/kaggle/data/input", "/kaggle/data"]:
        alt = f"{alt_root}/aptos2019-blindness-detection/test.csv"
        alt_img = f"{alt_root}/aptos2019-blindness-detection/test_images"
        if os.path.exists(alt):
            test_csv_path = alt
            test_img_root = alt_img
            break

df_test = pd.read_csv(test_csv_path)
test_dataset = ImageDataset(
    root=test_img_root,
    path_list=df_test.id_code.values,
    transform=test_transform,
)

print("Test csv:", test_csv_path)
print("Test img root:", test_img_root, "| exists:", os.path.isdir(test_img_root))
print("Test samples:", len(test_dataset))



Test csv: ../input/aptos2019-blindness-detection/test.csv
Test img root: ../input/aptos2019-blindness-detection/test_images | exists: True
Test samples: 367


In [5]:
from torch.utils.data import DataLoader
import pandas as pd

from torchvision.transforms import RandomHorizontalFlip, RandomRotation
from sklearn.model_selection import train_test_split
from sklearn.metrics import cohen_kappa_score
import numpy as np

val_indices = None
val_true = None
val_pred_raw = None
val_pred_mapped = None

# Change (score): replace coarse class->mean-label mapping with a monotone threshold calibration
# on the validation set using the model's expected value. This preserves the same model/loss/training,
# and only adjusts final discretization to better match ordinal QWK.
thresholds_ = None

if not have_external_weights:
    train_transform = Compose(
        [
            Resize((image_size, image_size), BICUBIC),
            RandomHorizontalFlip(p=0.5),
            RandomRotation(degrees=10),
            ToTensor(),
            Normalize(mean=[0.42, 0.22, 0.075], std=[0.27, 0.15, 0.081]),
        ]
    )

    train_csv_path = "../input/aptos2019-blindness-detection/train.csv"
    train_img_root = "../input/aptos2019-blindness-detection/train_images"
    if not os.path.exists(train_csv_path):
        for alt_root in ["/kaggle/input", "/kaggle/data/input", "/kaggle/data"]:
            alt = f"{alt_root}/aptos2019-blindness-detection/train.csv"
            alt_img = f"{alt_root}/aptos2019-blindness-detection/train_images"
            if os.path.exists(alt):
                train_csv_path = alt
                train_img_root = alt_img
                break

    df_train = pd.read_csv(train_csv_path)

    tr_ids, va_ids, tr_y, va_y = train_test_split(
        df_train.id_code.values,
        df_train.diagnosis.values,
        test_size=0.15,
        random_state=0,
        stratify=df_train.diagnosis.values,
    )

    train_dataset = ImageDataset(
        root=train_img_root,
        path_list=tr_ids,
        targets=tr_y,
        transform=train_transform,
    )
    val_dataset = ImageDataset(
        root=train_img_root,
        path_list=va_ids,
        targets=va_y,
        transform=test_transform,
    )

    batch_size_train = 16 if device.type == "cuda" else 8
    batch_size_val = 64
    num_workers = min(4, os.cpu_count() or 0)

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size_train,
        num_workers=num_workers,
        shuffle=True,
        drop_last=True,
        pin_memory=torch.cuda.is_available(),
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size_val,
        num_workers=num_workers,
        shuffle=False,
        drop_last=False,
        pin_memory=torch.cuda.is_available(),
    )

    best_model = EfficientNet(num_classes=5).to(device)
    best_model.train()

    opt = torch.optim.Adam(best_model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()

    max_epochs = 6
    for epoch in range(max_epochs):
        running = 0.0
        n = 0
        for xb, yb in train_loader:
            xb = xb.to(device, non_blocking=True)
            yb = yb.to(device, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            logits = best_model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            opt.step()
            running += float(loss.detach().cpu())
            n += 1

        best_model.eval()
        va_preds = []
        va_trues = []
        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(device, non_blocking=True)
                logits = best_model(xb)
                pred = torch.argmax(logits, dim=1).cpu().numpy().astype(int)
                va_preds.append(pred)
                va_trues.append(yb.cpu().numpy().astype(int))
        va_preds = np.concatenate(va_preds)
        va_trues = np.concatenate(va_trues)
        qwk = cohen_kappa_score(va_trues, va_preds, weights="quadratic")

        best_model.train()
        print(
            f"epoch {epoch+1}/{max_epochs} | loss={running/max(1,n):.4f} | steps={n} | val_qwk(raw_argmax)={qwk:.5f}"
        )

    best_model.eval()
    try:
        torch.save(best_model.state_dict(), model_path)
        print("Saved trained weights to", model_path)
    except Exception as e:
        print("WARNING: could not save trained weights:", e)

    # Fit thresholds on validation predictions (expected value of probabilities)
    best_model.eval()
    va_probs = []
    va_trues2 = []
    with torch.no_grad():
        for xb, yb in val_loader:
            xb = xb.to(device, non_blocking=True)
            logits = best_model(xb)
            probs = torch.softmax(logits, dim=1).cpu().numpy()
            va_probs.append(probs)
            va_trues2.append(yb.cpu().numpy().astype(int))
    va_probs = np.concatenate(va_probs, axis=0)
    va_trues2 = np.concatenate(va_trues2, axis=0)

    classes = np.arange(5, dtype=np.float32)
    va_exp = (va_probs * classes[None, :]).sum(axis=1)

    def _apply_thresholds(x, thresholds):
        # thresholds: 4 increasing cut points
        t = np.asarray(thresholds, dtype=np.float32)
        x = np.asarray(x, dtype=np.float32)
        out = np.zeros_like(x, dtype=np.int64)
        out += (x > t[0]).astype(np.int64)
        out += (x > t[1]).astype(np.int64)
        out += (x > t[2]).astype(np.int64)
        out += (x > t[3]).astype(np.int64)
        return out.clip(0, 4)

    # Simple coordinate descent on thresholds, monotone by construction, to maximize QWK on val
    # (keeps inference semantics; only maps continuous score -> 0..4).
    def _fit_thresholds_coordinate_descent(
        x, y, init=(0.5, 1.5, 2.5, 3.5), grid_step=0.02, iters=2
    ):
        x = np.asarray(x, dtype=np.float32)
        y = np.asarray(y, dtype=np.int64)
        t = np.array(init, dtype=np.float32)

        def score(t_):
            pred = _apply_thresholds(x, t_)
            return cohen_kappa_score(y, pred, weights="quadratic")

        best_score = score(t)
        for _ in range(iters):
            for j in range(4):
                # keep ordering with small margins
                lo = 0.0 if j == 0 else float(t[j - 1] + 1e-3)
                hi = 4.0 if j == 3 else float(t[j + 1] - 1e-3)
                if hi <= lo:
                    continue
                # local search around current threshold
                center = float(t[j])
                start = max(lo, center - 0.75)
                end = min(hi, center + 0.75)
                grid = np.arange(start, end + 1e-8, grid_step, dtype=np.float32)
                best_local = center
                for cand in grid:
                    t_try = t.copy()
                    t_try[j] = cand
                    s = score(t_try)
                    if s > best_score:
                        best_score = s
                        best_local = cand
                t[j] = best_local
        return t, float(best_score)

    thresholds_, qwk_thr = _fit_thresholds_coordinate_descent(va_exp, va_trues2)
    va_pred_thr = _apply_thresholds(va_exp, thresholds_)
    qwk_thr_check = cohen_kappa_score(va_trues2, va_pred_thr, weights="quadratic")
    print(
        "Validation thresholds_ (expected-value cutpoints):",
        [float(x) for x in thresholds_],
    )
    print(f"val_qwk(thresholded_expected)={qwk_thr_check:.5f}")



epoch 1/6 | loss=1.0520 | steps=175 | val_qwk(raw_argmax)=0.00000
epoch 2/6 | loss=0.9229 | steps=175 | val_qwk(raw_argmax)=0.00000
epoch 3/6 | loss=0.8728 | steps=175 | val_qwk(raw_argmax)=0.00000
epoch 4/6 | loss=0.8567 | steps=175 | val_qwk(raw_argmax)=0.66708
epoch 5/6 | loss=0.8559 | steps=175 | val_qwk(raw_argmax)=0.63761
epoch 6/6 | loss=0.8405 | steps=175 | val_qwk(raw_argmax)=0.68954
Saved trained weights to efficientNet_best.pth
Validation thresholds_ (expected-value cutpoints): [0.17999999225139618, 1.609999179840088, 2.0709986686706543, 3.5]
val_qwk(thresholded_expected)=0.74246


In [6]:
from torch.utils.data import DataLoader
import os

batch_size = 64
num_workers = min(4, os.cpu_count() or 0)
print("num_workers:", num_workers)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    num_workers=num_workers,
    shuffle=False,
    drop_last=False,
    pin_memory=torch.cuda.is_available(),
)



num_workers: 4


In [7]:
from tqdm import tqdm
from torch.nn import functional as F
import torch
import numpy as np

all_pred = []
all_exp = []  # for thresholding branch only
best_model.eval()
with torch.no_grad():
    for x, _ in tqdm(test_loader, total=len(test_loader)):
        x = x.to(device, non_blocking=True)

        y_pred1 = best_model(x)
        y_pred2 = best_model(x.flip(dims=(-1,)))
        curr_pred = 0.5 * (F.softmax(y_pred1, dim=-1) + F.softmax(y_pred2, dim=-1))

        # Keep original behavior (argmax) for the external-weights branch.
        all_pred += torch.argmax(curr_pred, dim=1).cpu().numpy().astype(int).tolist()

        if not have_external_weights:
            # Change (score): also compute expected value for threshold calibration (scratch-training only).
            cls = torch.arange(5, device=curr_pred.device, dtype=curr_pred.dtype)
            expv = (curr_pred * cls[None, :]).sum(dim=1)
            all_exp += expv.cpu().numpy().astype(np.float32).tolist()

if (not have_external_weights) and (thresholds_ is not None):
    # Apply calibrated thresholds to expected values to get final 0..4 labels.
    x = np.asarray(all_exp, dtype=np.float32)
    t = np.asarray(thresholds_, dtype=np.float32)
    pred = np.zeros_like(x, dtype=np.int64)
    pred += (x > t[0]).astype(np.int64)
    pred += (x > t[1]).astype(np.int64)
    pred += (x > t[2]).astype(np.int64)
    pred += (x > t[3]).astype(np.int64)
    all_pred = pred.clip(0, 4).astype(int).tolist()

print("Predictions:", len(all_pred))



100%|██████████| 6/6 [00:13<00:00,  2.18s/it]

Predictions: 367


In [8]:
import numpy as np
import pandas as pd

assert len(all_pred) == len(
    df_test
), f"Prediction length mismatch: {len(all_pred)} vs {len(df_test)}"

sub = pd.DataFrame({"id_code": df_test["id_code"].values})
sub["diagnosis"] = np.array(all_pred, dtype=int).clip(0, 4)

sub.to_csv("submission.csv", index=False)
print(sub.head())
print("Wrote submission.csv with shape:", sub.shape)



        id_code  diagnosis
0  b460ca9fa26f          0
1  6cee2e148520          1
2  ca6842bfcbc9          2
3  6cbc3dad809c          2
4  a9bc2f892cb3          0
Wrote submission.csv with shape: (367, 2)


In [9]:
import matplotlib

matplotlib.use("Agg")
_ = sub.hist()



In [10]:
import matplotlib
import pandas as pd
import os

matplotlib.use("Agg")

train_csv_path = "../input/aptos2019-blindness-detection/train.csv"
if not os.path.exists(train_csv_path):
    for alt_root in ["/kaggle/input", "/kaggle/data/input", "/kaggle/data"]:
        alt = f"{alt_root}/aptos2019-blindness-detection/train.csv"
        if os.path.exists(alt):
            train_csv_path = alt
            break

tr = pd.read_csv(train_csv_path)
_ = tr.hist()
print("Train label distribution:\n", tr["diagnosis"].value_counts().sort_index())

Train label distribution:
 diagnosis
0    1628
1     340
2     896
3     176
4     255
Name: count, dtype: int64
